# Day 25 - Worst-Cases Profile & Feature Importance Diagnostic

## Objective

Extract high-hazard worst-case misclassifications, analyze tree Gini feature importances, and document operational hazard mitigation protocols.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

def _find_project_root():
    curr = Path.cwd()
    for p in [curr] + list(curr.parents):
        if (p / "pyproject.toml").exists() or (p / "requirements.txt").exists():
            return p
    return curr

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

np.random.seed(42)
print("Environment initialized successfully. Project root:", project_root)

Environment initialized successfully. Project root: F:\Desktop\IIT G\Internship\Water Potability


## 1. Worst-Case Residual Extraction

Extract top 10 misclassified samples exhibiting highest prediction confidence divergence from ground truth.

In [2]:
import joblib
df_ts = pd.read_csv("data/processed/water_quality_timestamped.csv")
cut2 = int(len(df_ts) * 0.85)
test_set = df_ts.iloc[cut2:].copy()
feature_cols = ["ph", "Hardness", "Solids", "Chloramines", "Sulfate", "Conductivity", "Organic_carbon", "Trihalomethanes", "Turbidity"]
X_test = test_set[feature_cols].fillna(test_set[feature_cols].median())
y_test = test_set["Potability"]
pipe = joblib.load(project_root / "models" / "production" / "water_potability_pipeline_v1.joblib")
test_probs = pipe.predict_proba(X_test)[:, 1]
residuals = np.abs(y_test - test_probs)
worst_idx = np.argsort(residuals.values)[-10:]
df_worst = test_set.iloc[worst_idx][feature_cols + ["Potability"]].copy()
df_worst["Predicted_Prob"] = test_probs[worst_idx].round(4)
print("Top 3 Worst-Case Misclassifications:")
print(df_worst.head(3).to_string(index=False))

Top 3 Worst-Case Misclassifications:
      ph   Hardness       Solids  Chloramines    Sulfate  Conductivity  Organic_carbon  Trihalomethanes  Turbidity  Potability  Predicted_Prob
     NaN 188.536608 24711.414927     7.129520        NaN    555.548534       16.959269        56.038702   4.331691           1          0.3258
9.271355 181.259617 16540.979048     7.022499 309.238865    487.692788       13.228441              NaN   4.333953           1          0.3233
8.250830 184.385124 21616.633855     7.166734 305.177426    412.947116       13.791337        71.799129   2.604708           1          0.3224


## 2. Chemical Hazard Interaction Analysis

Identify combinations of high Chloramines and abnormal pH contributing to worst-case errors.

In [3]:
worst_ph_mean = df_worst["ph"].mean()
worst_chlor_mean = df_worst["Chloramines"].mean()
print(f"Worst Cases Average pH: {worst_ph_mean:.2f}, Average Chloramines: {worst_chlor_mean:.2f}")

Worst Cases Average pH: 8.45, Average Chloramines: 6.77


## 3. Tree Ensemble Gini Feature Importance Extraction

Extract normalized Gini impurity feature importances from champion Random Forest model.

In [4]:
rf_model = pipe.named_steps["classifier"]
importances = rf_model.feature_importances_
df_imp = pd.DataFrame({"Feature": feature_cols, "Gini_Importance": np.round(importances, 4)}).sort_values("Gini_Importance", ascending=False)
print("Champion Model Feature Importances:")
print(df_imp.to_string(index=False))

Champion Model Feature Importances:
        Feature  Gini_Importance
        Sulfate           0.1978
             ph           0.1656
       Hardness           0.1330
    Chloramines           0.1174
         Solids           0.1142
   Conductivity           0.0777
Trihalomethanes           0.0669
 Organic_carbon           0.0650
      Turbidity           0.0624


## 4. Top Predictor Dominance Quantification

Calculate cumulative importance of top 3 features (pH, Sulfate, Hardness).

In [5]:
top3_imp = df_imp.head(3)["Gini_Importance"].sum() * 100
print(f"Top 3 Features Account for {top3_imp:.1f}% of Model Gini Importance.")

Top 3 Features Account for 49.6% of Model Gini Importance.


## 5. Operational Hazard Mitigation Protocol

Formulate automated warning trigger when raw readings fall into worst-case parameter clusters.

In [6]:
print("Operational mitigation: Secondary laboratory validation triggered whenever pH < 6.0 or Sulfate > 400 mg/L.")

Operational mitigation: Secondary laboratory validation triggered whenever pH < 6.0 or Sulfate > 400 mg/L.


## 6. Feature Importance Diagnostic Plot

Render horizontal bar chart displaying Gini importance rankings across all 9 features.

In [7]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(df_imp["Feature"], df_imp["Gini_Importance"], color="#1f77b4", alpha=0.85)
ax.invert_yaxis()
ax.set_title("Day 25 Task 2: Champion Random Forest Gini Feature Importances", fontsize=11, pad=10)
ax.set_xlabel("Normalized Gini Importance")
for i, v in enumerate(df_imp["Gini_Importance"]):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=8, fontweight="bold")
plt.tight_layout()
fig_path = Path("Day_Wise_Tasks/Day_25_Task_2/outputs/figures/day25_task2_feature_importances.png")
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved feature importances plot to:", fig_path)

Saved feature importances plot to: Day_Wise_Tasks\Day_25_Task_2\outputs\figures\day25_task2_feature_importances.png


C:\Users\Pavan kumar\AppData\Local\Temp\ipykernel_12852\2173150849.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Worst Cases Profile Table Serialization

Export worst-case profile samples table to CSV.

In [8]:
tbl_path = Path("Day_Wise_Tasks/Day_25_Task_2/outputs/tables/day25_task2_worst_cases.csv")
tbl_path.parent.mkdir(parents=True, exist_ok=True)
df_worst.to_csv(tbl_path, index=False)
print("Saved worst cases profile table to:", tbl_path)

Saved worst cases profile table to: Day_Wise_Tasks\Day_25_Task_2\outputs\tables\day25_task2_worst_cases.csv


## 8. Hazard Profiling Gate Assertion

Assert that Gini importances sum to 1.0 within numerical precision.

In [9]:
assert abs(sum(importances) - 1.0) < 1e-4
print("Day 25 Task 2 Quality Gate: PASSED.")

Day 25 Task 2 Quality Gate: PASSED.


# Day 25 - Assumptions and Methodology

## Assumptions

1. Worst-case misclassifications identify specific environmental parameter combinations requiring secondary testing.
2. Gini importance highlights primary physicochemical drivers of potability classification.

## Day 25 Deliverable

Worst-cases profile table, Gini importance diagnostic plot, and operational hazard mitigation report.